# 11주차 — JOIN — 테이블 연결
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다.
어떤 셀이 오류로 막히거나 데이터가 엉켰다면, 아무 셀에서 `reset_db()`를 실행하면 처음 상태로 돌아갑니다.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, shutil, sqlite3, urllib.request
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"
DB_PATH = "mini_emr.sqlite"
WORK_PATH = DB_PATH   # 실습에 사용할 파일

con = None

def reset_db():
    """DB를 원본 상태로 되돌리고 다시 연결 (막히면 언제든 호출)"""
    global con
    if con is not None:
        con.close()
    try:
        urllib.request.urlretrieve(DB_URL, DB_PATH)
    except Exception as e:
        if not os.path.exists(DB_PATH):
            raise RuntimeError(
                "DB 다운로드 실패 — 왼쪽 파일 탭에 mini_emr.sqlite를 업로드한 뒤 이 셀을 다시 실행하세요."
            ) from e
        print("다운로드 실패 — 업로드되어 있는 파일을 사용합니다.")
    if WORK_PATH != DB_PATH:
        shutil.copyfile(DB_PATH, WORK_PATH)   # 원본 보존, 실습은 사본에서
    con = sqlite3.connect(WORK_PATH)
    con.execute("PRAGMA foreign_keys = ON")   # SQLite는 기본으로 꺼져 있음 — 반드시 켠다
    assert con.execute("PRAGMA foreign_keys").fetchone()[0] == 1

def q(sql):
    """조회(SELECT) 실행 → 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """데이터/구조를 바꾸는 SQL(INSERT/UPDATE/DELETE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

reset_db()
q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 1. INNER JOIN

In [ ]:
# 내원 + 환자 정보
q("""
SELECT v.visit_id, v.visit_date, v.department, p.sex, p.birth_date
FROM visit v
JOIN patient p ON v.patient_id = p.patient_id
LIMIT 8
""")

In [ ]:
# 3테이블: 내원 + 환자 + 담당의
q("""
SELECT v.visit_id, v.visit_date, p.sex AS 환자성별, s.role, s.department AS 소속과
FROM visit v
JOIN patient p ON v.patient_id = p.patient_id
JOIN staff s ON v.attending_id = s.staff_id
WHERE v.visit_type = '입원' ORDER BY v.visit_date DESC LIMIT 8
""")

## 2. 대표 질문 — 당뇨 환자들의 HbA1c (한 단계씩)

In [ ]:
# ① 같은 내원 안에서: 당뇨 진단이 붙은 그 내원의 HbA1c
q("""
SELECT d.kcd_code, l.test_name, l.result_value, l.collected_at
FROM diagnosis d
JOIN lab l ON l.visit_id = d.visit_id
WHERE d.kcd_code LIKE 'E11%' AND l.test_name = 'HbA1c'
ORDER BY l.collected_at
""")

①은 진단과 검사가 **같은 내원**에 있을 때만 잡습니다. 다른 내원의 검사까지 보려면 visit을 한 번 더 통과해 '같은 환자의 모든 내원'으로 확장합니다.

In [ ]:
# ② 환자 단위로 확장: 당뇨(E11) 환자의 모든 HbA1c 결과
q("""
SELECT DISTINCT v.patient_id, l.result_value AS hba1c, l.collected_at
FROM diagnosis d
JOIN visit v  ON d.visit_id = v.visit_id
JOIN visit v2 ON v2.patient_id = v.patient_id
JOIN lab l    ON l.visit_id = v2.visit_id
WHERE d.kcd_code LIKE 'E11%' AND l.test_name = 'HbA1c'
ORDER BY v.patient_id, l.collected_at
""")

## 3. LEFT JOIN — 없는 사람이 보인다

In [ ]:
# 당뇨 환자 전원 + HbA1c 검사 건수·최근 검사시각 (NULL = 미검사)
q("""
SELECT dm.patient_id,
       COUNT(l.lab_id) AS 검사건수,
       MAX(l.collected_at) AS 최근검사시각
FROM (SELECT DISTINCT v.patient_id
      FROM diagnosis d JOIN visit v ON d.visit_id = v.visit_id
      WHERE d.kcd_code LIKE 'E11%') dm
LEFT JOIN visit v2 ON v2.patient_id = dm.patient_id
LEFT JOIN lab l ON l.visit_id = v2.visit_id AND l.test_name = 'HbA1c'
GROUP BY dm.patient_id
""")

주의: `GROUP BY` 후에 집계함수 없이 `l.result_value` 같은 일반 컬럼을 SELECT하면 SQLite는 **임의의 한 행 값**을 돌려줍니다 — 최신값도 최대값도 아닙니다. 반드시 COUNT/MAX 등 집계함수로 의도를 명시하세요.

In [ ]:
# HbA1c를 한 번도 안 받은 당뇨 환자
q("""
SELECT DISTINCT v.patient_id
FROM diagnosis d JOIN visit v ON d.visit_id = v.visit_id
WHERE d.kcd_code LIKE 'E11%'
  AND v.patient_id NOT IN (
    SELECT v2.patient_id FROM lab l
    JOIN visit v2 ON l.visit_id = v2.visit_id
    WHERE l.test_name = 'HbA1c')
""")

이 NOT IN은 하위 명단에 NULL이 없어 안전하게 동작합니다 — NULL이 섞이면 어떻게 되는지는 **다음 주(서브쿼리)의 핵심 주제**입니다.

## 연습문제

In [ ]:
# [연습 Q1] 각 진단(dx_name)에 해당 환자의 성별을 붙여 출력하라
# 힌트: diagnosis→visit→patient
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] 뇌 MRI를 찍은 내원의 환자ID·촬영일·판독문을 출력하라
# 힌트: imaging + visit
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] 영상 검사를 한 번도 안 받은 환자는 몇 명인가?
# 힌트: LEFT JOIN … IS NULL 또는 NOT IN
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q4] 폐렴(J18) 내원에서 나간 약물 목록을 만들라
# 힌트: diagnosis + prescription, 내원 기준
q("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. 진단+성별**
```sql
SELECT d.dx_name, p.sex FROM diagnosis d
JOIN visit v ON d.visit_id = v.visit_id
JOIN patient p ON v.patient_id = p.patient_id;
```

**A2. 뇌 MRI**
```sql
SELECT v.patient_id, i.taken_date, i.report FROM imaging i
JOIN visit v ON i.visit_id = v.visit_id
WHERE i.modality = 'MRI' AND i.body_part = '뇌';
```

**A3. 영상 없는 환자 수**
```sql
-- 하위 명단에 NULL이 없어 NOT IN이 안전 (NULL 위험은 12주차에서)
SELECT COUNT(*) FROM patient p
WHERE p.patient_id NOT IN (
  SELECT v.patient_id FROM imaging i JOIN visit v ON i.visit_id = v.visit_id);
```

**A4. 폐렴 약물**
```sql
SELECT DISTINCT rx.drug_name FROM diagnosis d
JOIN prescription rx ON rx.visit_id = d.visit_id
WHERE d.kcd_code LIKE 'J18%';
```